# Notebook 2 — Exploration brute du pluviographe
## Station de Kara — Événements pluvieux 2004–2014

**Objectif de ce notebook :**  
Explorer le fichier brut `INTENSITES_DE_PLUIE_-KARA.xlsx` sans aucune transformation.  
On cherche à comprendre la structure réelle du fichier (11 feuilles, 5 blocs par feuille),  
vérifier la cohérence des mesures, calculer les intensités, et produire des statistiques  
et visualisations descriptives sur les données telles qu'elles sont.

**Ce notebook ne produit pas de fichier de sortie.** Il prépare la compréhension  
nécessaire au Notebook 3 (extraction et nettoyage propres).

---

## 0. Imports et configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.color":       "#e0e0e0",
    "grid.linewidth":   0.6,
    "font.size":        11,
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})

FICHIER = "INTENSITES_DE_PLUIE_-KARA.xlsx"
print("Imports OK")


Imports OK


## 1. Chargement brut du fichier

Le fichier contient **11 feuilles**, une par année de 2004 à 2014.  
On inspecte d'abord la liste des feuilles, puis le contenu brut de la feuille 2004  
pour comprendre la mise en page avant d'écrire le moindre code d'extraction.


In [2]:
from openpyxl import load_workbook

wb = load_workbook(FICHIER, read_only=True)
print(f"Feuilles disponibles : {wb.sheetnames}")
print(f"Nombre de feuilles   : {len(wb.sheetnames)}")
print(f"Période couverte     : {wb.sheetnames[0]} – {wb.sheetnames[-1]}")


ModuleNotFoundError: No module named 'openpyxl'

In [ ]:
# Affichage brut de la feuille 2004 — lignes 3 à 10 (en-têtes + premiers jours)
df_brut_2004 = pd.read_excel(FICHIER, sheet_name="2004", header=None)
print(f"Dimensions de chaque feuille : {df_brut_2004.shape[0]} lignes × {df_brut_2004.shape[1]} colonnes")
print()
print("Lignes 3 à 5 (en-têtes) :")
print(df_brut_2004.iloc[3:6, :12].to_string())
print()
print("Lignes 6 à 12 (10 premiers jours, colonnes 0 à 11) :")
df_brut_2004.iloc[6:13, 0:12]


## 2. Compréhension de la structure

Chaque feuille est identique en mise en page. Elle contient **5 blocs mensuels**  
côte à côte correspondant à la saison des pluies. Les mois de saison sèche  
(novembre à mai) ne sont pas enregistrés.

### Organisation des colonnes

| Mois | Col. DATES | Hauteur matin | Durée matin | Hauteur soir | Durée soir |
|---|:---:|:---:|:---:|:---:|:---:|
| Juin | 1 | 2 | 3 | 4 | 5 |
| Juillet | 7 | 8 | 9 | 10 | 11 |
| Août | 13 | 14 | 15 | 16 | 17 |
| Septembre | 19 | 20 | 21 | 22 | 23 |
| Octobre | 25 | 26 | 27 | 28 | 29 |

### Organisation des lignes

| Lignes | Contenu |
|---|---|
| 0 à 2 | Marges de présentation (vides) |
| 3 | Noms des mois |
| 4 | Périodes : `Du matin au soir` / `Du soir au lendemain` |
| 5 | Unités : `Hauteurs en mm` / `Durées en mn` |
| **6 à 36** | **Données — jours 1 à 31** |

### Structure d'un événement

Pour chaque jour, **deux événements** peuvent exister :
- **Matin au soir** : pluie tombée dans la journée
- **Soir au lendemain** : pluie tombée dans la nuit

Pour chaque événement : hauteur en mm + durée en minutes.  
L'intensité se calcule : `I (mm/h) = hauteur ÷ (durée / 60)`.  
Un événement n'existe que si hauteur **et** durée sont toutes deux renseignées.


In [ ]:
# Carte des colonnes — constante pour toutes les années
BLOCS_MOIS = {
    "Juin":      {"jour_col": 1,  "hm_col": 2,  "dm_col": 3,  "hs_col": 4,  "ds_col": 5},
    "Juillet":   {"jour_col": 7,  "hm_col": 8,  "dm_col": 9,  "hs_col": 10, "ds_col": 11},
    "Août":      {"jour_col": 13, "hm_col": 14, "dm_col": 15, "hs_col": 16, "ds_col": 17},
    "Septembre": {"jour_col": 19, "hm_col": 20, "dm_col": 21, "hs_col": 22, "ds_col": 23},
    "Octobre":   {"jour_col": 25, "hm_col": 26, "dm_col": 27, "hs_col": 28, "ds_col": 29},
}
MOIS_NUM = {"Juin": 6, "Juillet": 7, "Août": 8, "Septembre": 9, "Octobre": 10}

print("Carte des colonnes par mois :")
print(f"{'Mois':<12} {'DATES':>7} {'H_matin':>9} {'D_matin':>9} {'H_soir':>8} {'D_soir':>8}")
print("-" * 57)
for mois, cols in BLOCS_MOIS.items():
    print(f"{mois:<12} {cols['jour_col']:>7} {cols['hm_col']:>9} {cols['dm_col']:>9} "
          f"{cols['hs_col']:>8} {cols['ds_col']:>8}")


## 3. Extraction exploratoire des données

On extrait ici tous les événements pluvieux des 11 années.  
Chaque ligne du DataFrame résultant représente **un événement pluvieux unique**  
(un jour × une période : matin ou soir).

La logique est simple : pour chaque feuille, pour chaque bloc mensuel,  
on parcourt les jours 1 à 31 (lignes 6 à 36). Un événement est retenu  
si et seulement si la hauteur **et** la durée sont toutes deux non nulles.


In [ ]:
def extraire_annee(fichier, annee):
    """
    Extrait tous les événements pluvieux d'une feuille annuelle.
    
    Paramètres
    ----------
    fichier : str  — chemin vers le fichier XLSX
    annee   : int  — année à extraire (2004 à 2014)
    
    Retourne
    --------
    DataFrame avec colonnes :
    annee, mois, mois_nom, jour, periode, hauteur_mm, duree_min, intensite_mmh
    """
    df_yr = pd.read_excel(fichier, sheet_name=str(annee), header=None)
    data  = df_yr.iloc[6:37].reset_index(drop=True)  # jours 1 à 31

    records = []
    for mois_nom, cols in BLOCS_MOIS.items():
        for periode, h_col, d_col in [
            ("matin au soir",     cols["hm_col"], cols["dm_col"]),
            ("soir au lendemain", cols["hs_col"], cols["ds_col"]),
        ]:
            for day_idx in range(31):
                h = data.iloc[day_idx, h_col]
                d = data.iloc[day_idx, d_col]

                # Un événement existe si hauteur ET durée sont renseignées
                if pd.notna(h) and pd.notna(d):
                    records.append({
                        "annee":         annee,
                        "mois":          MOIS_NUM[mois_nom],
                        "mois_nom":      mois_nom,
                        "jour":          day_idx + 1,
                        "periode":       periode,
                        "hauteur_mm":    float(h),
                        "duree_min":     float(d),
                        "intensite_mmh": float(h) / (float(d) / 60),
                    })
    return pd.DataFrame(records)


# Extraction complète — 11 années
frames = [extraire_annee(FICHIER, yr) for yr in range(2004, 2015)]
df = pd.concat(frames, ignore_index=True)

# Colonne date complète
df["date"] = pd.to_datetime(
    df[["annee","mois","jour"]].rename(
        columns={"annee":"year","mois":"month","jour":"day"}),
    errors="coerce"
)

print(f"Total événements extraits : {len(df):,}")
print(f"Années couvertes          : {df['annee'].min()} – {df['annee'].max()}")
print(f"Valeurs NaN dans 'annee'  : {df['annee'].isna().sum()}  (attendu : 0)")
print(f"Hauteur maximale          : {df['hauteur_mm'].max():.1f} mm")
print(f"Durée maximale            : {df['duree_min'].max():.0f} min "
      f"({df['duree_min'].max()/60:.1f} h)")
print(f"Intensité maximale        : {df['intensite_mmh'].max():.1f} mm/h")
print()
df.head(10)


## 4. Qualité des données

In [ ]:
total = len(df)

print("=== Événements par année ===")
print(f"{'Année':<8} {'Nb événements':>15} {'% du total':>12}")
print("-" * 38)
for yr, grp in df.groupby("annee"):
    n = len(grp)
    print(f"{yr:<8} {n:>15,} {n/total*100:>11.1f}%")
print("-" * 38)
print(f"{'Total':<8} {total:>15,} {'100.0%':>12}")
print()
print("Note : baisse nette à partir de 2011 (79 événements contre 125 en 2010).")
print("Cause probable : problème technique du pluviographe ou lacunes de saisie.")


In [ ]:
print("=== Événements par mois (toutes années) ===")
print(f"{'Mois':<12} {'Nb événements':>15} {'% du total':>12}")
print("-" * 42)
for mois_nom in ["Juin","Juillet","Août","Septembre","Octobre"]:
    n = (df["mois_nom"] == mois_nom).sum()
    print(f"{mois_nom:<12} {n:>15,} {n/total*100:>11.1f}%")
print()

print("=== Répartition matin / soir ===")
for p, grp in df.groupby("periode"):
    print(f"  {p:<22} : {len(grp):>4} événements ({len(grp)/total*100:.1f}%)")


In [ ]:
# Événements à durée très courte (< 6 min) — vérification de cohérence physique
courts = df[df["duree_min"] < 6].sort_values("duree_min")
print(f"Événements avec durée < 6 minutes : {len(courts)}")
print()
print(courts[["annee","mois_nom","jour","periode",
               "hauteur_mm","duree_min","intensite_mmh"]].to_string(index=False))
print()
print(f"→ {len(courts[courts.annee == 2011])}/17 de ces événements sont en 2011,")
print("  ce qui renforce l'hypothèse d'un problème de saisie cette année-là.")
print("  Ces événements ne sont pas supprimés ici — leur traitement sera")
print("  décidé au Notebook 3 (seuil de durée minimale).")


In [ ]:
print("=== Statistiques globales ===")
print()
print("Hauteurs (mm) :")
print(df["hauteur_mm"].describe().round(2).to_string())
print()
print("Durées (min) :")
print(df["duree_min"].describe().round(1).to_string())
print()
print("Intensités (mm/h) :")
print(df["intensite_mmh"].describe().round(2).to_string())
print()
print(f"Percentile 90  : {df['intensite_mmh'].quantile(0.90):.1f} mm/h")
print(f"Percentile 95  : {df['intensite_mmh'].quantile(0.95):.1f} mm/h")
print(f"Percentile 99  : {df['intensite_mmh'].quantile(0.99):.1f} mm/h")


In [ ]:
# Distribution des durées et des hauteurs par tranches
bins_dur   = [0, 18, 30, 60, 120, 240, 480, 625]
labels_dur = ["≤18","19–30","31–60","61–120","121–240","241–480",">480"]
dist_dur   = pd.cut(df["duree_min"], bins=bins_dur, labels=labels_dur).value_counts().sort_index()

bins_h   = [0, 5, 10, 20, 30, 50, 75, 100, 165]
labels_h = ["0–5","5–10","10–20","20–30","30–50","50–75","75–100",">100"]
dist_h   = pd.cut(df["hauteur_mm"], bins=bins_h, labels=labels_h).value_counts().sort_index()

print("Distribution des durées :")
print(f"{'Tranche (min)':<14} {'Nb':>6} {'%':>7}")
print("-" * 30)
for label, cnt in dist_dur.items():
    print(f"{label:<14} {cnt:>6} {cnt/total*100:>6.1f}%")

print()
print("Distribution des hauteurs :")
print(f"{'Tranche (mm)':<14} {'Nb':>6} {'%':>7}")
print("-" * 30)
for label, cnt in dist_h.items():
    print(f"{label:<14} {cnt:>6} {cnt/total*100:>6.1f}%")


## 5. Visualisations

### 5.1 Nombre d'événements par année

In [ ]:
events_par_annee = df.groupby("annee").size()

fig, ax = plt.subplots(figsize=(11, 4))

couleurs = ["#FF7043" if yr >= 2011 else "#42A5F5"
            for yr in events_par_annee.index]

ax.bar(events_par_annee.index, events_par_annee.values,
       color=couleurs, width=0.7, edgecolor="white")
ax.axhline(events_par_annee.mean(), color="#1A237E", linestyle="--",
           linewidth=1.4, label=f"Moyenne {events_par_annee.mean():.0f} év/an")

for x, y in zip(events_par_annee.index, events_par_annee.values):
    ax.text(x, y + 1, str(y), ha="center", fontsize=9)

ax.set_title("Nombre d'événements pluvieux enregistrés par année (2004–2014)")
ax.set_xlabel("Année")
ax.set_ylabel("Nombre d'événements")
ax.set_xticks(range(2004, 2015))

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#42A5F5", label="2004–2010 (série complète)"),
    Patch(facecolor="#FF7043", label="2011–2014 (baisse qualité)"),
    plt.Line2D([0],[0], color="#1A237E", linestyle="--",
               label=f"Moyenne {events_par_annee.mean():.0f} év/an"),
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig("fig6_evenements_annee.png", dpi=150, bbox_inches="tight")
plt.show()
print("fig6_evenements_annee.png sauvegardée")


### 5.2 Répartition des événements par mois et heatmap annuelle

In [ ]:
events_par_mois = df.groupby("mois_nom").size().reindex(
    ["Juin","Juillet","Août","Septembre","Octobre"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barres par mois
ax1 = axes[0]
ax1.bar(events_par_mois.index, events_par_mois.values,
        color="#66BB6A", edgecolor="white", width=0.6)
ax1.set_title("Événements par mois\n(toutes années confondues)")
ax1.set_ylabel("Nombre d'événements")
ax1.set_xlabel("Mois")
for i, v in enumerate(events_par_mois.values):
    ax1.text(i, v + 1.5, str(v), ha="center", fontsize=10)

# Heatmap événements par année × mois
ax2 = axes[1]
pivot = df.groupby(["annee","mois_nom"]).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=["Juin","Juillet","Août","Septembre","Octobre"])

im = ax2.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax2.set_xticks(range(5))
ax2.set_xticklabels(["Juin","Juil","Août","Sept","Oct"])
ax2.set_yticks(range(len(pivot.index)))
ax2.set_yticklabels(pivot.index)
ax2.set_title("Heatmap : événements\npar année et par mois")
plt.colorbar(im, ax=ax2, label="Nb événements")

for i in range(len(pivot.index)):
    for j in range(5):
        val = pivot.values[i, j]
        ax2.text(j, i, str(val), ha="center", va="center",
                 fontsize=8, color="black" if val < 22 else "white")

plt.tight_layout()
plt.savefig("fig7_repartition_mois.png", dpi=150, bbox_inches="tight")
plt.show()
print("fig7_repartition_mois.png sauvegardée")


### 5.3 Distribution des durées et des hauteurs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Durées
ax1 = axes[0]
vals_dur = [dist_dur[l] / total * 100 for l in labels_dur]
ax1.bar(range(len(labels_dur)), vals_dur, color="#5C6BC0", edgecolor="white")
ax1.set_xticks(range(len(labels_dur)))
ax1.set_xticklabels([l + " min" for l in labels_dur], rotation=35, ha="right", fontsize=9)
ax1.set_title("Distribution des durées d'événements")
ax1.set_ylabel("% des événements")
for i, v in enumerate(vals_dur):
    ax1.text(i, v + 0.2, f"{v:.1f}%", ha="center", fontsize=8)

# Hauteurs
ax2 = axes[1]
vals_h = [dist_h[l] / total * 100 for l in labels_h]
ax2.bar(range(len(labels_h)), vals_h, color="#26A69A", edgecolor="white")
ax2.set_xticks(range(len(labels_h)))
ax2.set_xticklabels([l + " mm" for l in labels_h], rotation=30, ha="right", fontsize=9)
ax2.set_title("Distribution des hauteurs d'événements")
ax2.set_ylabel("% des événements")
for i, v in enumerate(vals_h):
    ax2.text(i, v + 0.2, f"{v:.1f}%", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("fig8_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("fig8_distributions.png sauvegardée")


### 5.4 Distribution des intensités

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogramme — intensités ≤ 200 mm/h pour lisibilité
ax1 = axes[0]
df_plot = df[df["intensite_mmh"] <= 200]
ax1.hist(df_plot["intensite_mmh"], bins=40, color="#AB47BC",
         edgecolor="white", linewidth=0.4)
ax1.axvline(df["intensite_mmh"].median(), color="#E53935", linestyle="--",
            linewidth=1.5, label=f"Médiane {df['intensite_mmh'].median():.1f} mm/h")
ax1.set_title("Distribution des intensités (≤ 200 mm/h)")
ax1.set_xlabel("Intensité (mm/h)")
ax1.set_ylabel("Nombre d'événements")
ax1.legend()

n_exclus = (df["intensite_mmh"] > 200).sum()
ax1.text(0.97, 0.95, f"{n_exclus} événement(s) > 200 mm/h\nexclu(s) du graphique",
         transform=ax1.transAxes, ha="right", va="top", fontsize=8,
         bbox=dict(boxstyle="round,pad=0.3", facecolor="#fff9c4", alpha=0.8))

# Boxplot par mois
ax2 = axes[1]
data_box = [df[df["mois_nom"] == m]["intensite_mmh"].values
            for m in ["Juin","Juillet","Août","Septembre","Octobre"]]
bp = ax2.boxplot(data_box, patch_artist=True, showfliers=True,
                 flierprops=dict(marker=".", markersize=3, alpha=0.4),
                 medianprops=dict(color="black", linewidth=1.5))
for patch in bp["boxes"]:
    patch.set_facecolor("#FFCC80")
ax2.set_xticklabels(["Juin","Juil","Août","Sept","Oct"])
ax2.set_title("Intensités par mois (mm/h)")
ax2.set_ylabel("Intensité (mm/h)")

plt.tight_layout()
plt.savefig("fig9_intensites.png", dpi=150, bbox_inches="tight")
plt.show()
print("fig9_intensites.png sauvegardée")


### 5.5 Relation hauteur — durée

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

couleurs_mois = {
    "Juin":      "#1565C0",
    "Juillet":   "#2E7D32",
    "Août":      "#F57F17",
    "Septembre": "#6A1B9A",
    "Octobre":   "#B71C1C",
}

for mois_nom, couleur in couleurs_mois.items():
    sous = df[df["mois_nom"] == mois_nom]
    ax.scatter(sous["duree_min"], sous["hauteur_mm"],
               color=couleur, alpha=0.45, s=18, label=mois_nom)

ax.set_xlabel("Durée (minutes)")
ax.set_ylabel("Hauteur (mm)")
ax.set_title("Relation hauteur – durée par événement et par mois (2004–2014)")
ax.legend(title="Mois", fontsize=9)

plt.tight_layout()
plt.savefig("fig10_hauteur_duree.png", dpi=150, bbox_inches="tight")
plt.show()
print("fig10_hauteur_duree.png sauvegardée")


### 5.6 Évolution des maxima annuels

In [ ]:
max_intensite = df.groupby("annee")["intensite_mmh"].max()
max_hauteur   = df.groupby("annee")["hauteur_mm"].max()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax1 = axes[0]
ax1.plot(max_intensite.index, max_intensite.values,
         color="#E53935", marker="o", markersize=6, linewidth=1.8)
ax1.axhline(max_intensite.mean(), color="#555", linestyle="--",
            linewidth=1.1, label=f"Moyenne {max_intensite.mean():.0f} mm/h")
ax1.set_title("Intensité maximale annuelle (mm/h)")
ax1.set_xlabel("Année")
ax1.set_ylabel("Intensité max (mm/h)")
ax1.set_xticks(range(2004, 2015))
ax1.tick_params(axis="x", rotation=45)
ax1.legend()

ax2 = axes[1]
ax2.plot(max_hauteur.index, max_hauteur.values,
         color="#1E88E5", marker="s", markersize=6, linewidth=1.8)
ax2.axhline(max_hauteur.mean(), color="#555", linestyle="--",
            linewidth=1.1, label=f"Moyenne {max_hauteur.mean():.0f} mm")
ax2.set_title("Hauteur maximale annuelle (mm)")
ax2.set_xlabel("Année")
ax2.set_ylabel("Hauteur max (mm)")
ax2.set_xticks(range(2004, 2015))
ax2.tick_params(axis="x", rotation=45)
ax2.legend()

plt.tight_layout()
plt.savefig("fig11_max_annuels.png", dpi=150, bbox_inches="tight")
plt.show()
print("fig11_max_annuels.png sauvegardée")


## 6. Anomalies et points d'attention

In [ ]:
print("=== Anomalies détectées lors de l'exploration ===")
print()

print("1. Baisse du nombre d'événements après 2010")
print("   2004–2010 : entre 104 et 126 événements/an")
print("   2011–2014 : entre 79 et 94 événements/an")
print("   Cause probable : problème technique du pluviographe ou lacunes de saisie.")
print("   → Mentionner comme limite dans le rapport final.")
print("   → Le Notebook 3 conservera ces données mais les traitera avec précaution.")
print()

print("2. Événements à durée très courte (< 6 min) — 17 événements")
print("   Concentrés en 2011 (14/17). Génèrent des intensités allant jusqu'à 570 mm/h.")
print("   Ces valeurs sont physiquement possibles pour des averses convectives")
print("   en Afrique de l'Ouest, mais leur fiabilité métrologique est discutable.")
print("   → Le Notebook 3 définira un seuil de durée minimale (≥ 6 min recommandé).")
print()

print("3. Événement exceptionnel — 163 mm en une seule occurrence")
idx_max_h = df["hauteur_mm"].idxmax()
ev = df.loc[idx_max_h]
print(f"   {int(ev.annee)}/{int(ev.mois):02d}/{int(ev.jour):02d} — {ev.periode}")
print(f"   Hauteur : {ev.hauteur_mm} mm | Durée : {ev.duree_min:.0f} min |"
      f" Intensité : {ev.intensite_mmh:.1f} mm/h")
print("   Valeur élevée mais cohérente pour la région.")
print()

print("4. Aucune valeur de hauteur ou de durée négative détectée.")
print(f"   Hauteur min observée : {df['hauteur_mm'].min()} mm")
print(f"   Durée min observée   : {df['duree_min'].min()} min")


## 7. Conclusion de l'exploration

### Ce que nous avons appris

Le fichier `INTENSITES_DE_PLUIE_-KARA.xlsx` contient **1 139 événements pluvieux**  
répartis sur 11 années (2004–2014), couvrant uniquement la saison des pluies  
(Juin à Octobre). L'extraction est correcte : aucune valeur d'année manquante,  
aucun enregistrement parasite.

| Indicateur | Valeur |
|---|---|
| Total événements | 1 139 |
| Années couvertes | 2004 – 2014 (11 ans) |
| Mois couverts | Juin – Octobre |
| Hauteur maximale | 163 mm |
| Durée maximale | 624 min (10 h 24) |
| Intensité maximale | 570 mm/h |
| Durée médiane | 54 min |
| Intensité médiane | 5,6 mm/h |

### Ce que le Notebook 3 devra faire

- Appliquer un seuil de durée minimale (≥ 6 min recommandé)  
- Produire un DataFrame propre au format `(date, periode, hauteur_mm, duree_min, intensite_mmh)`
- Exporter ce DataFrame en CSV pour le Notebook 4 (fusion et préparation)

---
*Fin du Notebook 2 — Exploration brute du pluviographe*
